# Introduction

In this notebook, we use the OpenClip model as transformer and trainform all the images into vectors, then feed the vectors to multiple ML algorithms.

## Convert Images into DataFrame

In [1]:
import torch
import os

from PIL import Image
import open_clip
import numpy as np
import pandas as pd

from nazi_symbols_classification.training.data_preparation import get_image_paths
from nazi_symbols_classification.training.evaluation import get_top1_evaluation
from sklearn.metrics import classification_report, confusion_matrix

/mnt/data/nazi-symbols-classification/venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [3]:
images = get_image_paths("/home/zhiwei/Projects/nazi-symbols-classification/datasets/nazi-symbols-detection", 
                         ("train", "test", "val"))

In [4]:
train_images = [image for image in images if image.startswith('/home/zhiwei/Projects/nazi-symbols-classification/datasets/nazi-symbols-detection/train')]
test_images = [image for image in images if image.startswith('/home/zhiwei/Projects/nazi-symbols-classification/datasets/nazi-symbols-detection/test')]
valid_images = [image for image in images if image.startswith('/home/zhiwei/Projects/nazi-symbols-classification/datasets/nazi-symbols-detection/val')]

In [8]:
len(train_images), len(test_images), len(valid_images)

(75369, 15018, 15376)

In [9]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

In [10]:
y_train.count("nazi-symbol"), y_test.count("nazi-symbol"), y_valid.count("nazi-symbol")

(4038, 205, 403)

In [11]:
def load_image(image_path):
    with torch.no_grad(), torch.autocast("cuda"):
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        return pd.DataFrame(image_features.float().numpy())

def preprocess_images(images):
    return pd.concat([load_image(image_path) for image_path in images], axis=0)

In [12]:
load_image(train_images[1])

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,-0.015738,0.093096,0.084752,0.112216,-0.000442,-0.065532,0.007091,-0.025539,-0.04525,0.040717,...,0.05709,-0.064796,-0.022393,0.011908,-0.00822,-0.018699,0.013882,0.019758,-0.008252,-0.018672


In [13]:
with open("training_data.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(train_images), 200):
    data = preprocess_images(train_images[i:i+200])
    with open("training_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

Palette images with Transparency expressed in bytes should be converted to RGBA images


In [14]:
training_data = pd.read_csv("training_data.csv")
training_data["label"] = y_train
training_data.to_csv("training_data.csv", index=False)

In [15]:
with open("validation_data.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(valid_images), 200):
    data = preprocess_images(valid_images[i:i+200])
    with open("validation_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

Palette images with Transparency expressed in bytes should be converted to RGBA images


In [16]:
validation_data = pd.read_csv("validation_data.csv")
validation_data["label"] = y_valid
validation_data.to_csv("validation_data.csv", index=False)

In [17]:
with open("test_data.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(test_images), 200):
    data = preprocess_images(test_images[i:i+200])
    with open("test_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

In [18]:
test_data = pd.read_csv("test_data.csv")
test_data["label"] = y_test
test_data.to_csv("test_data.csv", index=False)

## Load data for training

In [3]:
training_data = pd.read_csv("training_data.csv")
validation_data = pd.read_csv("validation_data.csv")
test_data = pd.read_csv("test_data.csv")

In [4]:
training_data.head()

,0,1,2,3,4,5,6,7,8,9,...,503,504,505,506,507,508,509,510,511,label
0,-0.036284,0.065120,-0.026674,0.037901,-0.023275,0.023836,0.002879,0.007602,0.011652,-0.048367,...,-0.037090,0.030496,0.035004,0.056767,0.044238,-0.025507,-0.046314,0.003570,-0.024306,non-nazi
1,-0.015738,0.093096,0.084752,0.112216,-0.000442,-0.065532,0.007091,-0.025539,-0.045250,0.040717,...,-0.064796,-0.022393,0.011908,-0.008220,-0.018699,0.013882,0.019758,-0.008252,-0.018672,non-nazi
2,0.018969,0.124155,-0.094642,-0.000600,-0.057754,-0.037017,0.009728,-0.046778,0.006461,-0.032716,...,0.084787,0.011991,-0.006625,-0.057902,-0.011498,0.018157,-0.002892,0.011497,0.022816,non-nazi
3,-0.004954,-0.224413,-0.049862,-0.022130,-0.010407,-0.041640,-0.035328,0.006612,-0.002796,0.007758,...,0.023775,-0.027235,-0.019263,0.046171,0.055021,-0.029527,0.041510,-0.026703,-0.059962,non-nazi
4,-0.013319,-0.039820,-0.079986,0.040090,-0.033282,0.037759,0.001027,0.018503,-0.020362,-0.111674,...,0.060417,-0.009748,-0.003844,-0.061079,0.012986,0.034320,-0.022465,-0.052805,0.005296,non-nazi


In [5]:
feature_columns = [str(i) for i in range(512)]
label_column = "label"

In [6]:
test_features = pd.concat([validation_data[feature_columns], test_data[feature_columns]])
test_labels = validation_data[label_column].tolist() + test_data[label_column].tolist()

In [7]:
%%time

# import the library
from sklearn.linear_model import LogisticRegression

# instantiate & fit
lr=LogisticRegression(max_iter=5000)
lr.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 6.72 s, sys: 2.95 s, total: 9.67 s
Wall time: 544 ms


LogisticRegression(max_iter=5000)

In [8]:
print("score on test: " + str(lr.score(test_features, test_labels)))
print(classification_report(lr.predict(test_features), test_labels))

score on test: 0.9961505560307955
              precision    recall  f1-score   support

 nazi-symbol       0.88      0.92      0.90       581
    non-nazi       1.00      1.00      1.00     29813

    accuracy                           1.00     30394
   macro avg       0.94      0.96      0.95     30394
weighted avg       1.00      1.00      1.00     30394



In [9]:
%%time

# import the library
from sklearn.linear_model import SGDClassifier

# instantiate & fit
sgd=SGDClassifier()
sgd.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 348 ms, sys: 52.5 ms, total: 400 ms
Wall time: 399 ms


SGDClassifier()

In [10]:
print("score on test: " + str(sgd.score(test_features, test_labels)))
print(classification_report(sgd.predict(test_features), test_labels))

score on test: 0.9965453707968678
              precision    recall  f1-score   support

 nazi-symbol       0.91      0.91      0.91       609
    non-nazi       1.00      1.00      1.00     29785

    accuracy                           1.00     30394
   macro avg       0.96      0.96      0.96     30394
weighted avg       1.00      1.00      1.00     30394



In [11]:
%%time

# import the library
from sklearn.neighbors import KNeighborsClassifier

# instantiate & fit
knn = KNeighborsClassifier(algorithm = 'brute', n_jobs=-1)
knn.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 136 ms, sys: 66.5 ms, total: 203 ms
Wall time: 202 ms


KNeighborsClassifier(algorithm='brute', n_jobs=-1)

In [12]:
print("score on test: " + str(knn.score(test_features, test_labels)))
print(classification_report(knn.predict(test_features), test_labels))

score on test: 0.9965124695663617
              precision    recall  f1-score   support

 nazi-symbol       0.94      0.89      0.91       636
    non-nazi       1.00      1.00      1.00     29758

    accuracy                           1.00     30394
   macro avg       0.97      0.95      0.96     30394
weighted avg       1.00      1.00      1.00     30394



In [13]:
%%time

# import the library
from sklearn.svm import LinearSVC

# instantiate & fit
svm=LinearSVC(C=0.0001)
svm.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 1.43 s, sys: 204 ms, total: 1.64 s
Wall time: 1.63 s


LinearSVC(C=0.0001)

In [14]:
print("score on test: " + str(svm.score(test_features, test_labels)))
print(classification_report(svm.predict(test_features), test_labels))

score on test: 0.9799960518523393
              precision    recall  f1-score   support

 nazi-symbol       0.00      0.00      0.00         0
    non-nazi       1.00      0.98      0.99     30394

    accuracy                           0.98     30394
   macro avg       0.50      0.49      0.49     30394
weighted avg       1.00      0.98      0.99     30394



Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.


In [15]:
%%time

# import the library
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
clf = DecisionTreeClassifier(min_samples_split=10,max_depth=3)
clf.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 14.8 s, sys: 52.8 ms, total: 14.8 s
Wall time: 14.8 s


DecisionTreeClassifier(max_depth=3, min_samples_split=10)

In [16]:
print("score on test: " + str(clf.score(test_features, test_labels)))
print(classification_report(clf.predict(test_features), test_labels))

score on test: 0.9876291373297361
              precision    recall  f1-score   support

 nazi-symbol       0.74      0.67      0.71       670
    non-nazi       0.99      0.99      0.99     29724

    accuracy                           0.99     30394
   macro avg       0.87      0.83      0.85     30394
weighted avg       0.99      0.99      0.99     30394



In [17]:
%%time

# import the library
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
bg=BaggingClassifier(DecisionTreeClassifier(min_samples_split=10,max_depth=3),max_samples=0.5,max_features=1.0,n_estimators=10)
bg.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 58 s, sys: 175 ms, total: 58.2 s
Wall time: 58.2 s


BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=3,
                                                   min_samples_split=10),
                  max_samples=0.5)

In [18]:
print("score on test: " + str(bg.score(test_features, test_labels)))
print(classification_report(bg.predict(test_features), test_labels))

score on test: 0.9894716062380733
              precision    recall  f1-score   support

 nazi-symbol       0.71      0.75      0.73       578
    non-nazi       1.00      0.99      0.99     29816

    accuracy                           0.99     30394
   macro avg       0.85      0.87      0.86     30394
weighted avg       0.99      0.99      0.99     30394



In [19]:
%%time

# import the library
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
adb = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2),n_estimators=100,learning_rate=0.5)
adb.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 17min 15s, sys: 2.95 s, total: 17min 18s
Wall time: 17min 18s


AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                   learning_rate=0.5, n_estimators=100)

In [20]:
print("score on test: " + str(adb.score(test_features, test_labels)))
print(classification_report(adb.predict(test_features), test_labels))

score on test: 0.9940777785089162
              precision    recall  f1-score   support

 nazi-symbol       0.79      0.90      0.84       534
    non-nazi       1.00      1.00      1.00     29860

    accuracy                           0.99     30394
   macro avg       0.89      0.95      0.92     30394
weighted avg       0.99      0.99      0.99     30394



In [21]:
%%time

# import the library
from sklearn.ensemble import GradientBoostingClassifier

# instantiate & fit
gbc = GradientBoostingClassifier(n_estimators=100)
gbc.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 24min 24s, sys: 104 ms, total: 24min 24s
Wall time: 24min 24s


GradientBoostingClassifier()

In [22]:
print("score on test: " + str(gbc.score(test_features, test_labels)))
print(classification_report(gbc.predict(test_features), test_labels))

score on test: 0.9938145686648681
              precision    recall  f1-score   support

 nazi-symbol       0.80      0.88      0.84       558
    non-nazi       1.00      1.00      1.00     29836

    accuracy                           0.99     30394
   macro avg       0.90      0.94      0.92     30394
weighted avg       0.99      0.99      0.99     30394



In [27]:
%%time

# import the library
from sklearn.ensemble import RandomForestClassifier

# instantiate & fit
rf = RandomForestClassifier(n_estimators=300,max_depth=3)
rf.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 2min 12s, sys: 62.5 ms, total: 2min 12s
Wall time: 2min 12s


RandomForestClassifier(max_depth=3, n_estimators=300)

In [28]:
print("score on test: " + str(rf.score(test_features, test_labels)))
print(classification_report(rf.predict(test_features), test_labels))

score on test: 0.9819701256827005
              precision    recall  f1-score   support

 nazi-symbol       0.10      1.00      0.18        60
    non-nazi       1.00      0.98      0.99     30334

    accuracy                           0.98     30394
   macro avg       0.55      0.99      0.59     30394
weighted avg       1.00      0.98      0.99     30394



In [29]:
%%time

# import the library
from sklearn.ensemble import VotingClassifier

evc=VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=5000)),
                                 ('rf', RandomForestClassifier(n_estimators=30,max_depth=3)),
                                 ('svm', LinearSVC(max_iter=5000))])
evc.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 21.3 s, sys: 3.63 s, total: 24.9 s
Wall time: 14.7 s


VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=5000)),
                             ('rf',
                              RandomForestClassifier(max_depth=3,
                                                     n_estimators=30)),
                             ('svm', LinearSVC(max_iter=5000))])

In [30]:
print("score on test: " + str(evc.score(test_features, test_labels)))
print(classification_report(evc.predict(test_features), test_labels))

score on test: 0.9959202474172534
              precision    recall  f1-score   support

 nazi-symbol       0.86      0.93      0.89       558
    non-nazi       1.00      1.00      1.00     29836

    accuracy                           1.00     30394
   macro avg       0.93      0.97      0.95     30394
weighted avg       1.00      1.00      1.00     30394

